# 01 - Exploratory Data Analysis

Notebook n?y ??c b? d? li?u Olist, m? t? shape/missing v? t?o c?c b?ng EDA ch?nh cho b?o c?o.

In [1]:
from src.etl.extract import load_olist_tables, summarize_tables
from src.etl.transform import build_order_features

tables = load_olist_tables()
summary = summarize_tables(tables)
summary

             table_name     rows  columns  duplicate_rows  missing_cells                                                                                                                                                                      columns_list
0  category_translation       71        2               0              0                                                                                                                              product_category_name, product_category_name_english
1             customers    99441        5               0              0                                                                                          customer_id, customer_unique_id, customer_zip_code_prefix, customer_city, customer_state
2           geolocation  1000163        5          261831              0                                                                                geolocation_zip_code_prefix, geolocation_lat, geolocation_lng, geolocation_city, geolocation_st

In [2]:
orders = tables['orders']
orders['order_status'].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2

In [3]:
features = build_order_features(tables)
features[['order_id', 'total_price', 'total_freight', 'review_score', 'delivery_days', 'delay_days', 'bad_review']].head()

                           order_id  total_price  total_freight  review_score  delivery_days  delay_days  bad_review
0  e481f51cbdc54678b7cc49136f2d6af7        29.99           8.72           4.0       8.436574   -7.107488         0.0
1  53cdb2fc8bc7dce0b6741e2150273451       118.70          22.76           4.0      13.782037   -5.355729         0.0
2  47770eb9100c2d0c44946d9cf07ec65d       159.90          19.22           5.0       9.394213  -17.245498         0.0
3  949d5b44dbf5de918fe9c16f97b45f8a        45.00          27.20           5.0      13.208750  -12.980069         0.0
4  ad21c59c0840e6cb83a9ceb5573f8159        19.90           8.72           5.0       2.873877   -9.238171         0.0

In [4]:
monthly = features.groupby('order_year_month').agg(orders=('order_id','nunique'), revenue=('total_price','sum')).reset_index()
monthly.tail(12)

   order_year_month  orders     revenue
13          2017-11    7544  1010271.37
14          2017-12    5673   743914.17
15          2018-01    7269   950030.36
16          2018-02    6728   844178.71
17          2018-03    7211   983213.44
18          2018-04    6939   996647.75
19          2018-05    6873   996517.68
20          2018-06    6167   865124.31
21          2018-07    6292   895507.22
22          2018-08    6512   854686.33
23          2018-09      16      145.00
24          2018-10       4        0.00

In [5]:
category = features.groupby('product_category_name_english').agg(orders=('order_id','nunique'), revenue=('total_price','sum'), avg_review=('review_score','mean'), delay_rate=('is_delayed','mean')).sort_values('revenue', ascending=False)
category.head(20)

                               orders     revenue  avg_review  delay_rate
product_category_name_english                                            
health_beauty                    8810  1260402.23    4.183800    0.087968
watches_gifts                    5584  1199042.79    4.073325    0.083811
bed_bath_table                   9384  1049531.99    3.974677    0.086424
sports_leisure                   7668   985965.36    4.177104      0.0759
computers_accessories            6679   915673.93    4.027188    0.075311
furniture_decor                  6350   733437.63    4.023730     0.08378
cool_stuff                       3589   632552.76    4.186164    0.067428
housewares                       5811   629327.06    4.154766    0.068663
auto                             3891   594688.67    4.091320    0.084297
garden_tools                     3479   484697.59    4.147816    0.078758
toys                             3841   479935.42    4.198398    0.073939
baby                             2870 

In [6]:
features.groupby('payment_type').agg(orders=('order_id','nunique'), revenue=('total_price','sum'), avg_review=('review_score','mean')).sort_values('orders', ascending=False)

              orders      revenue  avg_review
payment_type                                 
credit_card    74975  10725676.35    4.088537
boleto         19784   2391525.66    4.087136
voucher         3151    290675.87    4.005874
debit_card      1527    183630.85    4.169954
not_defined        3         0.00    1.666667
unknown            1       134.97    1.000000

In [7]:
features.groupby('review_group').agg(orders=('order_id','nunique'), avg_delay=('delay_days','mean'), delay_rate=('is_delayed','mean'))

              orders  avg_delay  delay_rate
review_group                               
bad            14449  -4.433917    0.286248
good           76054 -12.433843    0.034857
neutral         8170 -10.072088    0.107099